# 개별종목 조합I — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합I 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합I의 피처 값만 지정합니다.
import json

COMBINATION = 'I'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합I 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.5033,0.5012,0.0021,0.3003,0.3580,0.0781,0.3808,0.0542,0.1262
1,2,balanced,980,20150123,20150421,0.3957,0.3978,-0.0022,0.3493,0.3643,0.0592,0.3797,0.1816,0.2753
2,3,balanced,1210,20151228,20160328,0.3614,0.3762,-0.0147,0.3593,0.3595,0.0419,0.3709,0.3157,0.3441
3,4,balanced,1439,20161202,20170228,0.4635,0.4617,0.0018,0.3654,0.3845,0.1018,0.4071,0.1701,0.2785
4,5,balanced,1669,20171113,20180207,0.4169,0.3901,0.0268,0.3809,0.3932,0.1006,0.3941,0.2598,0.3381
5,6,balanced,1899,20181024,20190118,0.4146,0.3725,0.0421,0.4138,0.4230,0.1373,0.4182,0.5259,0.4458
6,7,balanced,2129,20190930,20191224,0.4727,0.4781,-0.0055,0.3551,0.3784,0.0966,0.4075,0.1720,0.2791
7,8,balanced,2359,20200902,20201130,0.4068,0.3476,0.0592,0.4045,0.4091,0.1138,0.4060,0.4517,0.4199
8,9,balanced,2589,20210806,20211105,0.3777,0.3914,-0.0137,0.3661,0.3830,0.0701,0.3796,0.2410,0.3148
9,10,balanced,2818,20220714,20221012,0.3454,0.3454,0.0000,0.3452,0.3488,0.0248,0.3620,0.2731,0.3174


,OOS 폴드 평균
accuracy,0.4124
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0156
macro_f1,0.3678
balanced_accuracy,0.3825
mcc,0.0846
pr_auc_macro_ovr,0.3920
down_recall,0.2735
core_harmonic_mean,0.3222


재실행 명령: python scripts/run_stock_model_experiment.py
